#01 - NHANES Data Pull
Project: Mets Risk Score - Multimodel Prediction in Young Adults
Data: NHANES 2017-March 2020 Pre-Pendemic Files
Purpose:  Download all required P_ tables from CDC, save raw .XPT files to Google Drive  


In [ ]:
# Cell 2: Mount Google Drive
# This connects your google drive to colab so files presist between sessions.
# Only need to Authorize once every session - a popup will apper

from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted sucessfully")

In [ ]:
# Cell 3: Defining Project Paths

import os
# Root
Drive_root = '/content/drive/MyDrive/mets-risk-score/'

# Sub-folders
data_raw  =  os.path.join(Drive_root, 'data/raw/')  # raw NHANES .XPT files
data_proc =  os.path.join(Drive_root, 'data/processed/') # merged/cleaned data
notebooks    = os.path.join(Drive_root, 'notebooks/')       # notebook backups
fig_dir      = os.path.join(Drive_root, 'outputs/figures/') # saved plots
model_dir    = os.path.join(Drive_root, 'outputs/models/')  # saved models
src_dir      = os.path.join(Drive_root, 'src/')             # Python modules

# Verify all folder exists
paths = {
    'Raw Data':    data_raw,
    'Processed':   data_proc,
    'Notebooks':   notebooks,
    'Figures':     fig_dir,
    'Models':      model_dir,
    'Source':      src_dir
}

print("Project folder status: \n")
for name, path in paths.items():
  status = "found" if os.path.exists(path) else "not found"
  print(f" {name:<12} {status}")
  print(f"            {path}")


In [ ]:
# Cell 4: Install Packages
# Run every session

!pip install nhanes miceforest xgboost lightgbm shap --quiet

In [4]:
# Cell 5: Libraries

# Data
import pandas as pd
import numpy as np

# NHANES data loader
from nhanes.load import load_NHANES_data

# System
import os
import warnings
warnings.filterwarnings('ignore')

In [12]:
# ── Cell 6 CORRECTED: NHANES P_ Tables with Verified URLs ─────────────────
#
# The pre-pandemic P_ files live at this CDC path:
# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/
#
# Key details confirmed from CDC directly:
# - Folder is /Public/2017/DataFiles/ for ALL P_ pre-pandemic files
# - File extension is lowercase .xpt
# - Filename prefix is P_ followed by the table name

BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"

NHANES_TABLES = {

    # ── DOMAIN 1: Demographics ─────────────────────────────────────────────
    'DEMO':   'P_DEMO.xpt',

    # ── DOMAIN 2: Secondary Biochemistry ───────────────────────────────────
    'BIOPRO': 'P_BIOPRO.xpt',

    # ── DOMAIN 3: Complete Blood Count ─────────────────────────────────────
    'CBC':    'P_CBC.xpt',

    # ── DOMAIN 4: Dietary Intake ───────────────────────────────────────────
    'DR1TOT': 'P_DR1TOT.xpt',
    'DR2TOT': 'P_DR2TOT.xpt',

    # ── DOMAIN 5: Physical Activity ────────────────────────────────────────
    'PAQ':    'P_PAQ.xpt',

    # ── DOMAIN 6: Sleep ────────────────────────────────────────────────────
    'SLQ':    'P_SLQ.xpt',

    # ── DOMAIN 7: Mental Health (PHQ-9) ────────────────────────────────────
    'DPQ':    'P_DPQ.xpt',

    # ── DOMAIN 8: Smoking ──────────────────────────────────────────────────
    'SMQ':    'P_SMQ.xpt',

    # ── DOMAIN 9: Alcohol ──────────────────────────────────────────────────
    'ALQ':    'P_ALQ.xpt',

    # ── DOMAIN 10: Body Measures ───────────────────────────────────────────
    'BMX':    'P_BMX.xpt',

    # ── TARGET VARIABLE COMPONENTS (MSSS only — excluded from predictors) ──
    'TRIGLY': 'P_TRIGLY.xpt',
    'HDL':    'P_HDL.xpt',
    'GLU':    'P_GLU.xpt',
    'BPX':    'P_BPXO.xpt',
}

print(f"📋 Tables defined: {len(NHANES_TABLES)}")
print(f"   Base URL: {BASE_URL}\n")
for name, file in NHANES_TABLES.items():
    print(f"   {name:<10} {BASE_URL + file}")

📋 Tables defined: 15
   Base URL: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/

   DEMO       https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.xpt
   BIOPRO     https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_BIOPRO.xpt
   CBC        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_CBC.xpt
   DR1TOT     https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DR1TOT.xpt
   DR2TOT     https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DR2TOT.xpt
   PAQ        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_PAQ.xpt
   SLQ        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SLQ.xpt
   DPQ        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DPQ.xpt
   SMQ        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.xpt
   ALQ        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_ALQ.xpt
   BMX        https://wwwn.cdc.gov/Nchs/Data/Nhanes/Pu

In [ ]:
# ── Cell 7 CORRECTED: Download All Tables ──────────────────────────────────

import requests
import os

def download_nhanes_table(table_name, file_name, base_url, save_dir):
    """
    Downloads a single NHANES XPT file from CDC and saves to Drive.
    Validates the file is real binary data, not an HTML error page.
    """
    save_path = os.path.join(save_dir, file_name)

    # Check if a valid file already exists
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            header = f.read(200)
        is_html = b'<html' in header.lower() or b'<!doc' in header.lower()

        if not is_html:
            size_mb = os.path.getsize(save_path) / (1024 * 1024)
            print(f"   ⏭️  {table_name:<10} already exists ({size_mb:.1f} MB) — skipping")
            return save_path
        else:
            os.remove(save_path)
            print(f"   🗑️  {table_name:<10} bad file removed — re-downloading")

    # Download
    url = base_url + file_name
    print(f"   ⬇️  {table_name:<10} downloading...", end=" ")

    try:
        response = requests.get(url, timeout=120)

        if response.status_code == 200:
            # Check it's not an HTML error page
            is_html = b'<html' in response.content[:200].lower()
            if is_html:
                print(f"❌ CDC returned error page")
                print(f"      URL tried: {url}")
                return None

            with open(save_path, 'wb') as f:
                f.write(response.content)
            size_mb = os.path.getsize(save_path) / (1024 * 1024)
            print(f"✅ saved ({size_mb:.1f} MB)")
            return save_path
        else:
            print(f"❌ HTTP {response.status_code}")
            print(f"      URL tried: {url}")
            return None

    except Exception as e:
        print(f"❌ Error: {e}")
        return None


# ── Run downloads ───────────────────────────────────────────────────────────
print("🔽 Downloading NHANES 2017-March 2020 Pre-Pandemic Tables")
print(f"   Saving to: {data_raw}\n")

downloaded = {}
failed = []

for table_name, file_name in NHANES_TABLES.items():
    path = download_nhanes_table(table_name, file_name, BASE_URL, data_raw)
    if path:
        downloaded[table_name] = path
    else:
        failed.append(table_name)

print(f"\n{'─'*50}")
print(f"✅ Downloaded : {len(downloaded)} tables")
if failed:
    print(f"❌ Failed     : {failed}")
else:
    print(f"❌ Failed     : none")# ── Cell 7 CORRECTED: Download All Tables ──────────────────────────────────

import requests
import os

def download_nhanes_table(table_name, file_name, base_url, save_dir):
    """
    Downloads a single NHANES XPT file from CDC and saves to Drive.
    Validates the file is real binary data, not an HTML error page.
    """
    save_path = os.path.join(save_dir, file_name)

    # Check if a valid file already exists
    if os.path.exists(save_path):
        with open(save_path, 'rb') as f:
            header = f.read(200)
        is_html = b'<html' in header.lower() or b'<!doc' in header.lower()

        if not is_html:
            size_mb = os.path.getsize(save_path) / (1024 * 1024)
            print(f"   ⏭️  {table_name:<10} already exists ({size_mb:.1f} MB) — skipping")
            return save_path
        else:
            os.remove(save_path)
            print(f"   🗑️  {table_name:<10} bad file removed — re-downloading")

    # Download
    url = base_url + file_name
    print(f"   ⬇️  {table_name:<10} downloading...", end=" ")

    try:
        response = requests.get(url, timeout=120)

        if response.status_code == 200:
            # Check it's not an HTML error page
            is_html = b'<html' in response.content[:200].lower()
            if is_html:
                print(f"❌ CDC returned error page")
                print(f"      URL tried: {url}")
                return None

            with open(save_path, 'wb') as f:
                f.write(response.content)
            size_mb = os.path.getsize(save_path) / (1024 * 1024)
            print(f"✅ saved ({size_mb:.1f} MB)")
            return save_path
        else:
            print(f"❌ HTTP {response.status_code}")
            print(f"      URL tried: {url}")
            return None

    except Exception as e:
        print(f"❌ Error: {e}")
        return None


# ── Run downloads ───────────────────────────────────────────────────────────
print("🔽 Downloading NHANES 2017-March 2020 Pre-Pandemic Tables")
print(f"   Saving to: {data_raw}\n")

downloaded = {}
failed = []

for table_name, file_name in NHANES_TABLES.items():
    path = download_nhanes_table(table_name, file_name, BASE_URL, data_raw)
    if path:
        downloaded[table_name] = path
    else:
        failed.append(table_name)

print(f"\n{'─'*50}")
print(f"✅ Downloaded : {len(downloaded)} tables")
if failed:
    print(f"❌ Failed     : {failed}")
else:
    print(f"❌ Failed     : none")

In [ ]:
# ── Cell 8: Load and Verify All Tables ─────────────────────────────────────
#
# Now we load each XPT file into a pandas DataFrame and do a quick
# sanity check — rows, columns, and a peek at the first few rows.
# This confirms the download worked and we understand what we have.

print("📊 Verifying downloaded tables\n")
print(f"{'Table':<10} {'Rows':>8} {'Columns':>10}   Key Columns (first 6)")
print("─" * 70)

tables = {}

for table_name, file_name in NHANES_TABLES.items():
    path = os.path.join(data_raw, file_name)

    try:
        df = pd.read_sas(path, format='xport', encoding='utf-8')
        tables[table_name] = df

        # Show first 6 column names for a quick preview
        cols_preview = ', '.join(df.columns[:6].tolist())
        print(f"{table_name:<10} {len(df):>8,} {len(df.columns):>10}   {cols_preview}")

    except Exception as e:
        print(f"{table_name:<10} ❌ Error loading: {e}")

print("─" * 70)
print(f"\n✅ {len(tables)} tables loaded into memory")
print(f"   All tables share 'SEQN' — the participant ID used to merge them")

In [ ]:
# ── Cell 9: Save as CSV for Easy Access Later ──────────────────────────────
#
# XPT (SAS format) files are slow to read repeatedly.
# We convert each table to CSV once and save to data/raw/.
# Future notebooks will read the CSVs — much faster.

print("💾 Converting XPT files to CSV format\n")

for table_name, df in tables.items():
    csv_path = os.path.join(data_raw, f"{table_name}.csv")

    if os.path.exists(csv_path):
        print(f"   ⏭️  {table_name}.csv already exists — skipping")
        continue

    df.to_csv(csv_path, index=False)
    size_kb = os.path.getsize(csv_path) / 1024
    print(f"   ✅ {table_name}.csv saved ({size_kb:.0f} KB)")

print(f"\n✅ All CSVs saved to: {data_raw}")